# 0. Overview
- Dataset: Breast Cancer
- Task: Classification
- Model: Logistic Regression
- Loss function: Binary Cross Entrophy


In [3]:
from sklearn.datasets import load_breast_cancer

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import torch.optim as optim

# 1. Data Collection

In [ ]:
data = load_breast_cancer()

In [5]:
type(data)

sklearn.utils._bunch.Bunch

In [7]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# 2. Data Preprocessing

In [8]:
df['class'] = data.target
cols = ['mean radius', 'mean texture','mean smoothness', 'mean compactness','mean concave points',
        'worst radius', 'worst texture','worst smoothness', 'worst compactness','worst concave points', 'class']

In [9]:
df[cols].isnull().sum()

mean radius             0
mean texture            0
mean smoothness         0
mean compactness        0
mean concave points     0
worst radius            0
worst texture           0
worst smoothness        0
worst compactness       0
worst concave points    0
class                   0
dtype: int64

In [11]:
data = torch.from_numpy(df[cols].values).float()

In [12]:
X = data[:, :-1]
y = data[:, -1:]
print(X.shape)
print(y.shape)

torch.Size([569, 10])
torch.Size([569, 1])


# 3. Model Selection

## Hyperparameter

In [13]:
n_epochs = 20000
learning_rate = 1e-2
print_interval = 10000

In [16]:
class MyModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.output_dim = output_dim
        self.input_dim = input_dim

        self.linear = nn.Linear(input_dim, output_dim)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.act(self.linear(x))
        return y

In [23]:
model = MyModel(input_dim = X.shape[-1], output_dim = y.shape[-1])
loss_fn = nn.BCELoss()  # loss function : Binary Cross Entropy
optimizer = optim.SGD(model.parameters(), lr = learning_rate)

# 4. Model Training

In [24]:
for i in range(n_epochs):
    y_hat = model.forward(X)
    loss = loss_fn(y_hat, y)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if (i+1) % print_interval == 0:
        print(f'Epoch {i}: loss = {loss}')

Epoch 9999: loss = 0.2748315632343292
Epoch 19999: loss = 0.22719015181064606


# 5. Model Evaluation

In [ ]:
# 이진 분류의 값을 갖도록 0.5 이상인 값은 1로 판단하도록 한 후 성능을 평가해본 결과 91.2%의 준수한 성능을 보여주었다.
n_correct = (y == (y_hat > 0.5)).sum()
n_total = float(y.size(0))
print(f'Accuracy: {n_correct / n_total}')

Accuracy: 0.9191564321517944
